# Bronze: raw ingestion

Read CSV files from the Unity Catalog landing volume and write them to the Bronze Delta table. This starter uses a full refresh: each run replaces the table with the current CSV files in the volume.

In [ ]:
from pyspark.sql import functions as F

CATALOG = "superpower"
BRONZE_SCHEMA = "bronze"
LANDING_VOLUME = "landing"
SOURCE_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/{LANDING_VOLUME}"
BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.csv_raw"

csv_paths = sorted(
    entry.path
    for entry in dbutils.fs.ls(SOURCE_PATH)
    if not entry.isDir() and entry.path.lower().endswith(".csv")
)

if not csv_paths:
    raise ValueError(f"No CSV files found in {SOURCE_PATH}")

bronze_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("mode", "PERMISSIVE")
    .load(csv_paths)
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_ingested_at", F.current_timestamp())
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}")
(
    bronze_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

print(f"Wrote {len(csv_paths)} CSV file(s) to {BRONZE_TABLE}")
display(spark.table(BRONZE_TABLE).limit(20))